# NovaPay Day 8: API Packaging and Production Review

This notebook documents the final FastAPI packaging stage for the NovaPay fraud detection project. It is the primary reference for the API implementation: model loading, payload validation, feature engineering expectations, local testing, Swagger review, and Docker deployment.

The trained model is consumed as an existing artifact. This notebook does not retrain or alter the model; it explains how the API wraps the model for real-time transaction scoring.


## Final Deliverables

The production-ready API package includes the following files and folders:

```text
api/
  main.py              # FastAPI app, lifespan startup, endpoints
  schemas.py           # Pydantic request and response models
  model_loader.py      # Model and threshold loading
  scoring.py           # Feature engineering and prediction response logic
  requirements.txt     # API dependency mirror
models/day6/
  best_advanced_model.joblib
reports/artifacts/day8/
  sample_request.json
  sample_response.json
  api_validation_notes.md
README.md
PROJECT_SUMMARY.md
TESTING_GUIDE.md
FINAL_SUBMISSION_CHECKLIST.md
Dockerfile
requirements.txt
```


## Model Loading Contract

The API expects the trained Day 6 model at:

```text
models/day6/best_advanced_model.joblib
```

At application startup, `api.model_loader.ModelRegistry` loads this artifact and an optional threshold file if one is present beside the model. If the model is unavailable, `/health` still responds with `model_loaded: false`, and `/score` returns a clear HTTP `503` instead of failing silently.


In [ ]:
from pathlib import Path

model_path = Path("models/day6/best_advanced_model.joblib")
print("Model path:", model_path)
print("Model exists:", model_path.exists())
if not model_path.exists():
    print("Fallback: generate the model by running 06_advanced_models.ipynb first.")

## Expected Request Schema

The `/score` endpoint accepts one transaction at a time. Required fields are:

`transaction_id`, `customer_id`, `timestamp`, `home_country`, `source_currency`, `dest_currency`, `channel`, `amount_src`, `amount_usd`, `fee`, `exchange_rate_src_to_dest`, `device_id`, `new_device`, `ip_address`, `ip_country`, `location_mismatch`, `ip_risk_score`, `kyc_tier`, `account_age_days`, `device_trust_score`, `chargeback_history_count`, `risk_score_internal`, `txn_velocity_1h`, `txn_velocity_24h`, and `corridor_risk`.

Implemented validation covers non-empty string fields, parseable datetimes, non-negative amounts and counts, a positive exchange rate, booleans for device/location flags, and risk scores bounded between 0 and 1. The API intentionally treats IP address, country, currency, channel, and KYC values as non-empty strings rather than enforcing fixed allow-lists.


In [ ]:
sample_payload = {
    "transaction_id": "TX12345",
    "customer_id": "CUST1001",
    "timestamp": "2026-07-03T10:15:00Z",
    "home_country": "GB",
    "source_currency": "GBP",
    "dest_currency": "NGN",
    "channel": "mobile_app",
    "amount_src": 750.0,
    "amount_usd": 950.0,
    "fee": 8.5,
    "exchange_rate_src_to_dest": 1850.25,
    "device_id": "DEV-7788",
    "new_device": True,
    "ip_address": "203.0.113.10",
    "ip_country": "NG",
    "location_mismatch": True,
    "ip_risk_score": 0.86,
    "kyc_tier": "tier_2",
    "account_age_days": 24,
    "device_trust_score": 0.22,
    "chargeback_history_count": 1,
    "risk_score_internal": 0.84,
    "txn_velocity_1h": 7,
    "txn_velocity_24h": 28,
    "corridor_risk": 0.78,
}

sample_payload

## Example Invalid Payload

This invalid example shows common validation failures: an empty transaction ID, a negative amount, and a risk score outside the accepted 0-to-1 range.

In [ ]:
invalid_payload = sample_payload.copy()
invalid_payload["transaction_id"] = ""
invalid_payload["amount_usd"] = -25
invalid_payload["ip_risk_score"] = 1.5

invalid_payload

## Run the API Locally

From the project root, install dependencies and start the service:

```bash
pip install -r requirements.txt
uvicorn api.main:app --reload
```

Useful local URLs:

```text
http://127.0.0.1:8000/health
http://127.0.0.1:8000/docs
```


In [ ]:
import requests

health_url = "http://127.0.0.1:8000/health"
# Run after starting uvicorn in a terminal.
# requests.get(health_url, timeout=10).json()

## Score One Transaction

The `/score` endpoint accepts one validated transaction payload and returns a prediction, fraud probability, confidence score, operational decision, explanation, and model version.

Use the Swagger UI at `http://127.0.0.1:8000/docs` or post `reports/artifacts/day8/sample_request.json` with `curl`, Postman, or the Python `requests` library.


In [ ]:
score_url = "http://127.0.0.1:8000/score"
# Run after starting uvicorn and ensuring the Day 6 model exists.
# response = requests.post(score_url, json=sample_payload, timeout=10)
# response.status_code, response.json()

## Example Response Format

The sample request included in this project currently produces the following response with the bundled model artifact:

```json
{
  "transaction_id": "TX12345",
  "prediction": "fraud",
  "fraud_probability": 0.9987,
  "confidence_score": 0.9987,
  "decision": "hold_and_investigate",
  "reason": "Transaction flagged for high transaction velocity, high IP risk score, low device trust score, location mismatch, previous chargeback history, high corridor risk, unusual high internal risk score.",
  "model_version": "day6_best_advanced_model"
}
```


## Docker Commands

Build the image from the project root:

```bash
docker build -t novapay-fraud-api .
```

Run the container:

```bash
docker run -p 8000:8000 novapay-fraud-api
```

Then open the API documentation:

```text
http://127.0.0.1:8000/docs
```


## Production Validation Reflection

The most important validations for real-time scoring are those that protect the model from malformed, impossible, or unsafe values: non-empty identifiers, valid timestamps, non-negative money and count fields, positive exchange rates, boolean flags, and bounded risk scores. These checks keep bad payloads out of the prediction pipeline and make API behavior easier for reviewers and downstream systems to trust.


## Assessment

Primary API artifacts:

- Sample request: `reports/artifacts/day8/sample_request.json`
- Sample response: `reports/artifacts/day8/sample_response.json`
- Validation notes: `reports/artifacts/day8/api_validation_notes.md`
- Testing guide: `TESTING_GUIDE.md`

The sample request demonstrates a high-risk transaction, and the sample response demonstrates the final response structure returned by the deployed service.
